# LeWM training on Google Colab
Run the cells top to bottom. **Before starting:** Runtime > Change runtime type > GPU (T4 is enough).

Steps: 1) check GPU, 2) clone repo, 3) install deps, 4) download PushT data (~13 GB compressed), 5) smoke test, 6) full training, 7) save checkpoints.

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU - switch the runtime type to GPU first"
!df -h /content | tail -1

## 1. Clone the repo

In [ ]:
%cd /content
!git clone https://github.com/arshagarwal/le-wm.git
%cd /content/le-wm
!git log --oneline | head -3

## 2. Install dependencies
This takes a few minutes. If pip reports a torch version change, that is fine.

In [ ]:
!pip install -q "stable-worldmodel[train]" hdf5plugin h5py hf_transfer
!apt-get -qq install -y zstd > /dev/null
import stable_worldmodel, stable_pretraining, hdf5plugin
from stable_worldmodel.data.format import list_formats
print("formats:", list_formats())
assert "hdf5" in list_formats(), "hdf5 format not registered - check hdf5plugin/h5py install"
print("deps OK")

## 3. Download and decompress the PushT dataset
The dataset is an `.h5.zst` archive (~13 GB). It goes under `$STABLEWM_HOME/datasets/`, which is where this version of stable-worldmodel looks.

In [ ]:
import os
os.environ["STABLEWM_HOME"] = "/content/stablewm"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
DATA_DIR = "/content/stablewm/datasets"
os.makedirs(DATA_DIR, exist_ok=True)

!hf download quentinll/lewm-pusht --repo-type dataset --local-dir /content/dl_pusht
!ls -lh /content/dl_pusht

In [ ]:
# decompress; handles either a plain zstd'd .h5 or a zstd'd tar
!zstd -d -f /content/dl_pusht/pusht_expert_train.h5.zst -o /content/dl_pusht/out.bin
with open("/content/dl_pusht/out.bin", "rb") as f:
    magic = f.read(8)
if magic.startswith(b"\x89HDF"):
    !mv /content/dl_pusht/out.bin {DATA_DIR}/pusht_expert_train.h5
else:
    !tar -xf /content/dl_pusht/out.bin -C {DATA_DIR} && rm /content/dl_pusht/out.bin
!rm -f /content/dl_pusht/pusht_expert_train.h5.zst
!ls -lh {DATA_DIR}
!df -h /content | tail -1

In [ ]:
# sanity check: dataset loads and shows its columns
import stable_worldmodel as swm
ds = swm.data.load_dataset("pusht_expert_train.h5", frameskip=5, num_steps=4,
                           keys_to_load=["pixels", "action", "proprio", "state"],
                           keys_to_cache=["action", "proprio", "state"])
print(len(ds), "samples")
print({k: tuple(getattr(v, "shape", ())) for k, v in ds[0].items()})

## 4. Smoke test (about 1 minute)
Runs 5 train batches and 2 val batches to confirm everything works before the full run.
bf16 is used on GPUs that support it (A100/L4); T4 falls back to fp16 mixed precision.

In [ ]:
import torch
PRECISION = "bf16" if torch.cuda.is_bf16_supported() else "16-mixed"
print("precision:", PRECISION)

COMMON = (
    "data=pusht data.dataset.name=pusht_expert_train.h5 "
    f"trainer.precision={PRECISION} num_workers=2 wandb.enabled=False"
)
!cd /content/le-wm && STABLEWM_HOME=/content/stablewm python train.py {COMMON} \
    trainer.max_epochs=1 +trainer.limit_train_batches=5 +trainer.limit_val_batches=2 \
    output_model_name=smoke subdir=smoke

## 5. Full training
Defaults: 100 epochs, batch size 128, lr 5e-5. Colab sessions can disconnect, so the next cell mounts Google Drive and writes checkpoints there. If the session disconnects, re-run the Drive cell, the resume cell, and this cell: training resumes from the last finished epoch (same `subdir`).
If you hit an out-of-memory error, add `loader.batch_size=64`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/lewm", exist_ok=True)
os.environ["STABLEWM_HOME"] = "/content/drive/MyDrive/lewm"
# Lightning writes resumable ckpts under SPT_CACHE_DIR; keep them on Drive too
os.environ["SPT_CACHE_DIR"] = "/content/drive/MyDrive/lewm/spt_cache"
os.makedirs("/content/drive/MyDrive/lewm/spt_cache", exist_ok=True)
# dataset stays on local disk (fast); symlink it into the Drive-backed home
os.makedirs("/content/drive/MyDrive/lewm/datasets", exist_ok=True)
!ln -sf /content/stablewm/datasets/pusht_expert_train.h5 /content/drive/MyDrive/lewm/datasets/pusht_expert_train.h5
!ls -l /content/drive/MyDrive/lewm/datasets

In [ ]:
# Resume support: train.py resumes from <STABLEWM_HOME>/checkpoints/<subdir>/lewm_weights.ckpt,
# but Lightning writes last.ckpt under SPT_CACHE_DIR. Copy the newest one into place (no-op on a fresh run).
import glob, shutil
run_dir = "/content/drive/MyDrive/lewm/checkpoints/pusht_run1"
os.makedirs(run_dir, exist_ok=True)
cands = glob.glob("/content/drive/MyDrive/lewm/spt_cache/runs/*/*/*/checkpoints/last.ckpt")
if cands:
    newest = max(cands, key=os.path.getmtime)
    shutil.copy(newest, f"{run_dir}/lewm_weights.ckpt")
    print("will resume from", newest)
else:
    print("no previous run found - starting fresh")

In [ ]:
!cd /content/le-wm && STABLEWM_HOME=/content/drive/MyDrive/lewm python train.py {COMMON} \
    output_model_name=lewm subdir=pusht_run1

## 5b. CLAW latent-action run (optional)
Adds the CLAW latent action model + adversarial regularizer (`model=lewm_claw`). **This needs the CLAW code pushed to GitHub before the clone cell runs** (the clone only gets committed code). Extra metrics logged per step: `adv_loss`, `ground_loss`, `delta` (GRL scale, ramps 0 -> 0.999 over the run), `z_std` and `z_erank` (collapse check; both near 0 = collapsed), `adv_rel` (adversary error / baseline variance; near 1 = no leakage, near 0 = z leaks the next frame).
`delta` follows `max_epochs`, so keep `max_epochs` equal to the real run length. CLAW runs use their own Lightning cache dir (`spt_cache_claw`), so resuming a CLAW run never picks up the baseline's checkpoint (and vice versa). To resume after a disconnect, re-run the Drive cell and the dev cell below.

In [ ]:
# CLAW smoke test (about 1 minute)
!cd /content/le-wm && SPT_CACHE_DIR=/content/spt_cache_smoke STABLEWM_HOME=/content/stablewm python train.py {COMMON} model=lewm_claw \
    trainer.max_epochs=1 +trainer.limit_train_batches=5 +trainer.limit_val_batches=2 \
    output_model_name=claw_smoke subdir=claw_smoke

In [ ]:
# CLAW short run on Drive (reduced settings; raise max_epochs/img_size/batch for real runs).
# Resume: train.py looks for checkpoints/<subdir>/<output_model_name>_weights.ckpt; copy the newest CLAW last.ckpt there.
import glob, shutil
CLAW_CACHE = "/content/drive/MyDrive/lewm/spt_cache_claw"
claw_dir = "/content/drive/MyDrive/lewm/checkpoints/claw_dev"
os.makedirs(CLAW_CACHE, exist_ok=True); os.makedirs(claw_dir, exist_ok=True)
cands = glob.glob(f"{CLAW_CACHE}/runs/*/*/*/checkpoints/last.ckpt")
if cands:
    shutil.copy(max(cands, key=os.path.getmtime), f"{claw_dir}/claw_dev_weights.ckpt")
    print("will resume CLAW run from", max(cands, key=os.path.getmtime))
else:
    print("no previous CLAW run found - starting fresh")
!cd /content/le-wm && SPT_CACHE_DIR={CLAW_CACHE} STABLEWM_HOME=/content/drive/MyDrive/lewm python train.py {COMMON} model=lewm_claw \
    trainer.max_epochs=3 +trainer.limit_train_batches=100 +trainer.limit_val_batches=20 \
    img_size=112 loader.batch_size=32 \
    output_model_name=claw_dev subdir=claw_dev

## 6. Check outputs
Model weights: `MyDrive/lewm/checkpoints/lewm/weights_epoch_N.pt` (written every epoch; `lewm` = `output_model_name`).
Resumable Lightning checkpoints: `MyDrive/lewm/spt_cache/runs/.../checkpoints/last.ckpt`.

In [ ]:
!ls -lh /content/drive/MyDrive/lewm/checkpoints/lewm
!find /content/drive/MyDrive/lewm/spt_cache -name '*.ckpt' | head

## Optional: weights & biases logging
Add `wandb.enabled=True wandb.config.entity=<your_entity> wandb.config.project=<your_project>` to the training command and run `!wandb login` first.